# 03 · Quantize int8 + so sánh 1 tầng vs 2 tầng + đo tốc độ CPU

Chỉ dùng **CPU** (onnxruntime CPU) - cố ý không dùng GPU để số đo gần với laptop. Kaggle: Accelerator → None (4 nhân).
Colab: chạy luôn trên runtime T4 vừa dùng cho 01, 02 (CPU cũng 2 nhân như runtime CPU, lại khỏi tải lại VNTS).
Input: code + VNTS + output notebook 01 (models/detector_*.onnx) + 02 (models/signnet*.onnx), nằm sẵn trong Drive.
Secret `HF_TOKEN` (quyền write) để đẩy model lên HF Hub của chính bạn; không có thì bỏ qua bước đẩy.
Thời gian trên Colab: quantize ~3 phút, đo ~5 phút (+ ~5 phút nếu runtime mới phải tải lại VNTS).

In [ ]:
import glob, json, os, shutil, sys

IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/dashcam/__init__.py", recursive=True)[0].rsplit("/dashcam", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vn-dashcam-vision" if IN_COLAB else os.path.abspath("vn-dashcam-vision")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vn-dashcam-vision {REPO}
    else:  # đã clone ở lần chạy trước -> lấy bản code mới nhất
        !git -C {REPO} pull -q
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vn-dashcam-vision" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in (["/kaggle/input"] if IN_KAGGLE else []) + [WORK, REPO]:
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


# Colab: đọc hàng nghìn ảnh từ Google Drive rất chậm -> dataset để ở ổ cục bộ /content (mất khi ngắt,
# chạy lại cell là có), còn model / kết quả / checkpoint YOLO thì nằm trong WORK trên Drive.
DATA = "/content/data" if IN_COLAB else "datasets"
DS = f"{DATA}/vnts"
os.makedirs(DATA, exist_ok=True)


def get_vnts():
    """VNTS giống notebook 01: kagglehub (không cần tài khoản) -> mirror HF."""
    raw = f"{DATA}/vnts_raw"
    if glob.glob(f"{raw}/**/classes.txt", recursive=True):
        return raw
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
        v = secret(k)
        if v:
            os.environ[k] = v
    try:
        import kagglehub
        return kagglehub.dataset_download("maitam/vietnamese-traffic-signs")
    except Exception as e:
        print("kagglehub lỗi:", repr(e)[:300])
    try:  # mirror không có split_dataset/ -> test chia lại, khác bản Kaggle
        from huggingface_hub import snapshot_download
        snapshot_download("thanhhiepvos/vietnam_traffic_sign", repo_type="dataset", local_dir=raw,
                          allow_patterns=["archive/*"])
        return raw
    except Exception as e:
        print("HF lỗi:", repr(e)[:300])
    raise RuntimeError(f"không tải được VNTS, xem notebook 01 (có thể giải nén zip từ Kaggle vào {raw})")


# test + ảnh calibration: runtime còn dữ liệu từ notebook 01 thì dùng luôn, không thì tải + chuẩn bị lại
# (prepare_vnts chia train/val/test cố định nên ra đúng tập test như lúc train)
if IN_KAGGLE:
    VNTS = [d for d in glob.glob("/kaggle/input/*") if "traffic" in d.lower()][0]
elif not os.path.exists(f"{DS}/yolo_ncls/data.yaml"):
    VNTS = get_vnts()
os.makedirs("models", exist_ok=True)
if IN_KAGGLE:
    for f in glob.glob("/kaggle/input/**/models/*", recursive=True):
        shutil.copy(f, "models/")
else:  # Colab: models/ của notebook 01, 02 đã nằm trong WORK
    find("models/detector_1cls.onnx", "chạy notebook 01 trước")
    find("models/signnet.onnx", "chạy notebook 02 trước")
!pip install -q onnx onnxruntime
if IN_KAGGLE or not os.path.exists(f"{DS}/yolo_ncls/data.yaml"):
    !python -m dashcam.prepare_vnts --src {VNTS} --out {DS}
THREADS = min(4, os.cpu_count())  # Colab free chỉ 2 nhân: ép 4 luồng thì chậm hơn, số đo sai
print("CPU", os.cpu_count(), "nhân -> đo với", THREADS, "luồng")
!ls -la models

In [ ]:
!python -m dashcam.export --yolo models/detector_1cls.onnx --yolo models/detector_ncls.onnx \
    --calib-images {DS}/yolo_ncls/images/train --imgsz 640 --out models

## mAP@0.5 trên test (ảnh tĩnh) + ms/ảnh với THREADS luồng CPU

In [ ]:
!python -m dashcam.evaluate --images {DS}/yolo_ncls/images/test --labels {DS}/yolo_ncls/labels/test \
    --models models --class-stats {DS}/class_stats.json --threads {THREADS} --limit 300 --out results
print(open("results/eval.md", encoding="utf-8").read())

## 1 luồng (giống thiết bị nhúng yếu)

In [ ]:
!python -m dashcam.evaluate --images {DS}/yolo_ncls/images/test --labels {DS}/yolo_ncls/labels/test \
    --models models --threads 1 --limit 100 --out results_1thread
print(open("results_1thread/eval.md", encoding="utf-8").read())

In [ ]:
from huggingface_hub import HfApi

token = secret("HF_TOKEN")
if not token:
    print("thiếu secret HF_TOKEN -> chưa đẩy lên HF Hub (models/ vẫn nằm trong Drive)")
else:
    api = HfApi(token=token)
    HF_REPO = f"{api.whoami()['name']}/vn-dashcam-vision"  # repo dưới tài khoản của chính token
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(folder_path="models", repo_id=HF_REPO, allow_patterns=["*.onnx", "*.json"])
    api.upload_folder(folder_path="results", repo_id=HF_REPO, path_in_repo="results")
    print(f"https://huggingface.co/{HF_REPO}")